# Bronze Layer — Raw Parse
Read full 3.3 GB access.log from HDFS, apply regex, write Parquet partitioned by date.
Minimal transformation: only cast status (int) and bytes (long). Everything else stays as string.

In [1]:
import os, sys
os.environ['SPARK_HOME'] = '/usr/local/spark-3.5.0-bin-hadoop3'
# Tell HDFS client to authenticate as root (same user that created the FS via hdfs-namenode container)
os.environ['HADOOP_USER_NAME'] = 'root'
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python')
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip')
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType

spark = (
    SparkSession.builder
    .appName('02_bronze')
    .master('local[4]')
    .config('spark.hadoop.fs.defaultFS', 'hdfs://hdfs-namenode:9000')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .config('spark.hadoop.hadoop.security.authentication', 'simple')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('HADOOP_USER_NAME:', os.environ.get('HADOOP_USER_NAME'))


Spark version: 3.5.0
HADOOP_USER_NAME: root


In [2]:
# Read full 3.3 GB log from HDFS
raw = spark.read.text('hdfs://hdfs-namenode:9000/raw/access-logs/access.log')
print(f'Raw line count: {raw.count():,}')


Raw line count: 10,365,152


In [3]:
# Nginx combined log regex (validated in Phase 1: 99.999% match rate)
REGEX = (
    r'^(\S+) \S+ \S+ \[([^\]]+)\] '
    r'"(\S+) (\S+) ([^"]+)" (\d{3}) (\S+) '
    r'"([^"]*)" "([^"]*)"(?: "([^"]*)")?'
)

bronze = (
    raw
    .select(
        F.regexp_extract('value', REGEX, 1).alias('ip'),
        F.regexp_extract('value', REGEX, 2).alias('ts_raw'),
        F.regexp_extract('value', REGEX, 3).alias('method'),
        F.regexp_extract('value', REGEX, 4).alias('path'),
        F.regexp_extract('value', REGEX, 5).alias('protocol'),
        F.regexp_extract('value', REGEX, 6).cast(IntegerType()).alias('status'),
        F.when(
            F.regexp_extract('value', REGEX, 7) == '-', None
        ).otherwise(
            F.regexp_extract('value', REGEX, 7).cast(LongType())
        ).alias('bytes'),
        F.regexp_extract('value', REGEX, 8).alias('referrer'),
        F.regexp_extract('value', REGEX, 9).alias('user_agent'),
        F.regexp_extract('value', REGEX, 10).alias('xff'),
    )
    .filter(F.col('ip') != '')
    .withColumn(
        'log_date',
        F.to_date(
            F.regexp_extract('ts_raw', r'^(\d{2}/\w+/\d{4})', 1),
            'dd/MMM/yyyy'
        )
    )
)

bronze.printSchema()
bronze.show(3, truncate=80)


root
 |-- ip: string (nullable = true)
 |-- ts_raw: string (nullable = true)
 |-- method: string (nullable = true)
 |-- path: string (nullable = true)
 |-- protocol: string (nullable = true)
 |-- status: integer (nullable = true)
 |-- bytes: long (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_agent: string (nullable = true)
 |-- xff: string (nullable = true)
 |-- log_date: date (nullable = true)

+------------+--------------------------+------+--------------------------------------------------------------------------------+--------+------+-----+-----------------------------------+--------------------------------------------------------------------------------+---+----------+
|          ip|                    ts_raw|method|                                                                            path|protocol|status|bytes|                           referrer|                                                                      user_agent|xff|  log_date|
+----------

In [4]:
# Write Bronze Parquet to HDFS, partitioned by log_date
bronze.write.mode('overwrite').partitionBy('log_date').parquet(
    'hdfs://hdfs-namenode:9000/data/bronze/access_logs'
)
print('Bronze write complete.')

# Verify: read back and count
verify = spark.read.parquet('hdfs://hdfs-namenode:9000/data/bronze/access_logs')
row_count = verify.count()
print(f'Bronze row count (from Parquet): {row_count:,}')

# Show partition dates present in the data
verify.select('log_date').distinct().orderBy('log_date').show(50)


Bronze write complete.
Bronze row count (from Parquet): 10,365,077
+----------+
|  log_date|
+----------+
|2019-01-22|
|2019-01-23|
|2019-01-24|
|2019-01-25|
|2019-01-26|
+----------+



In [5]:
spark.stop()
print('Bronze layer done.')


Bronze layer done.
